# Pixel-wise

In [14]:
import cv2
import numpy as np

def l1_distance(x,y):
    return abs(x-y)
def l2_distance(x,y):
    return (x-y)**2

def pixel_wise_matching(left_img_path,right_img_path,disparity_range=16,compute_type='l1',save_result=True):
    print(f"Compute disparity map using pixel-wise-matching with {compute_type.upper()} distance.")
    # 1. Read the left and right images
    left_img = cv2.imread(left_img_path, cv2.IMREAD_GRAYSCALE)
    right_img = cv2.imread(right_img_path, cv2.IMREAD_GRAYSCALE)
    if left_img is None or right_img is None:
        raise FileNotFoundError("Could not read one or both images. Please check the file paths.")
    # convert to float32 for distance computation
    left_img = left_img.astype(np.float32)
    right_img = right_img.astype(np.float32)

    # 2. initalize height and width of the images
    height, width = left_img.shape[:2]

    # 3. Initialize a zero matrix for depth
    depth = np.zeros((height,width),np.uint8)

    # set max cost based on compute type
    max_cost = 255 if compute_type == 'l1' else 255**2

    # scale factor for visualization
    scale_factor = 255.0 / disparity_range

    # 4. Iterate over each pixel (h,w)
    for h in range(height):
        for w in range(width):
            best_disparity = 0
            min_cost = float('inf')
            for d in range(disparity_range):
                if w-d < 0:
                    cost = max_cost
                else:
                    if compute_type == 'l1':
                        cost = l1_distance(left_img[h,w], right_img[h,w-d])
                    elif compute_type == 'l2':
                        cost = l2_distance(left_img[h,w], right_img[h,w-d])
                    else:
                        raise ValueError("Invalid compute type. Use 'l1' or 'l2'.")
                if cost < min_cost:
                    min_cost = cost
                    best_disparity = d
            depth[h,w] = int(best_disparity*scale_factor)
    if save_result:
        cv2.imwrite(f"disparity_map_pixel_wise_{compute_type}_gray.png", depth)
        color_map = cv2.applyColorMap(depth, cv2.COLORMAP_JET)
        cv2.imwrite(f"disparity_map_pixel_wise_{compute_type}_color.png", color_map)
        print("Done.")
    return depth

In [ ]:
%cd ../data

In [ ]:
!gdown --id 14gf8bcym_lTcvjZQmg8kwq3aXkENBxMQ
!unzip tsukuba.zip

In [ ]:
left_img_path = "..\\data\\left_water.png"
right_img_path = "..\\data\\right_water.png"
disparity_range = 16
pixel_wise_result_l1 = pixel_wise_matching(left_img_path, right_img_path, disparity_range, compute_type='l1', save_result=True)
pixel_wise_result_l2 = pixel_wise_matching(left_img_path, right_img_path, disparity_range, compute_type='l2', save_result=True)

# window based matching

In [17]:
def window_based_matching(left_img_path,right_img_path,disparity_range=16,compute_type='l1',kernel_size=5,save_result=True):
    print(f"Compute disparity map using window-based matching with {compute_type.upper()} distance.")
    left_img = cv2.imread(left_img_path, cv2.IMREAD_GRAYSCALE)
    right_img = cv2.imread(right_img_path, cv2.IMREAD_GRAYSCALE)

    if left_img is None or right_img is None:
        raise FileNotFoundError("Could not read one or both images. Please check the file paths.")
    # convert to float32 for distance computation
    left_img = left_img.astype(np.float32)
    right_img = right_img.astype(np.float32)
    # initalize height and width of the images
    height, width = left_img.shape[:2]
    # Initialize a zero matrix for depth
    depth = np.zeros((height,width),np.uint8)
    # calculate kernel half
    kernel_half = int((kernel_size)/2)
    # set max cost based on compute type
    max_cost_pixel = 255 if compute_type == 'l1' else 255**2
    max_cost_window = max_cost_pixel * (kernel_size**2)
    # scale factor for visualization
    scale_factor = 255.0 / disparity_range
    # Iterate over each pixel (h,w)
    for h in range(kernel_half,height-kernel_half):
        for w in range(kernel_half,width-kernel_half):
            best_disparity = 0
            min_total_cost = float('inf')
            # calculate sum of costs for each disparity
            for d in range(disparity_range):
                total_cost_window = 0.0
                for v in range(-kernel_half,kernel_half+1):
                    for u in range(-kernel_half,kernel_half+1):
                        if(w+u-d)<0:
                            pixel_cost = max_cost_pixel
                        else:
                            left_pixel_value = left_img[h+v,w+u]
                            right_pixel_value = right_img[h+v,w+u-d]

                            if compute_type == 'l1':
                                pixel_cost = l1_distance(left_pixel_value, right_pixel_value)
                            elif compute_type == 'l2':
                                pixel_cost = l2_distance(left_pixel_value, right_pixel_value)
                            else:
                                raise ValueError("Invalid compute type. Use 'l1' or 'l2'.")
                        total_cost_window += pixel_cost
                if total_cost_window < min_total_cost:
                    min_total_cost = total_cost_window
                    best_disparity = d
            depth[h,w] = int(best_disparity*scale_factor)
    if save_result:
        cv2.imwrite(f"disparity_map_window_based_{compute_type}_gray.png", depth)
        color_map = cv2.applyColorMap(depth, cv2.COLORMAP_JET)
        cv2.imwrite(f"disparity_map_window_based_{compute_type}_color.png", color_map)
        print("Done.")
    return depth


In [ ]:
left_img_path = "..\\data\\Aloe\\Aloe_left_1.png"
right_img_path = "..\\data\\Aloe\\Aloe_right_1.png"
disparity_range=16
kernel_size=5

In [22]:

window_based_result_l1 = window_based_matching(left_img_path, right_img_path, disparity_range, compute_type='l1', kernel_size=kernel_size, save_result=True)
window_based_result_l2 = window_based_matching(left_img_path, right_img_path, disparity_range, compute_type='l2', kernel_size=kernel_size, save_result=True)

Compute disparity map using window-based matching with L1 distance.
Done.
Compute disparity map using window-based matching with L2 distance.
Done.


# Window based matching with Consine Similarity

In [3]:
import cv2
import numpy as np
def cosine_similarity(x,y):
    numerator = np.dot(x,y)
    denominator = np.linalg.norm(x) * np.linalg.norm(y)
    return numerator / denominator if denominator != 0 else 0.0

In [5]:
def window_based_matching_cosine_similarity(left_img_path,right_img_path,disparity_range=64,kernel_size=3,save_result=True):
    # 1. read images as grayscale and convert to float32
    left_img = cv2.imread(left_img_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    right_img = cv2.imread(right_img_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    if left_img is None or right_img is None:
        raise ValueError("Could not read one of the images. Please check the file paths.")
    # 2. get image dimensions
    height, width = left_img.shape[:2]
    # 3. initialize disparity map
    depth = np.zeros((height,width),np.uint8)
    # 4. Calculate kernel_half
    kernel_half = int((kernel_size - 1) / 2)
    scale_factor = 255.0 / disparity_range

    for h in range(kernel_half,height-kernel_half):
        for w in range(kernel_half,width-kernel_half):
            best_disparity = 0
            max_similarity = -1
            
            left_window = left_img[h-kernel_half:h+kernel_half+1, w-kernel_half:w+kernel_half+1].flatten()

            for d in range(disparity_range):
                right_window_start_w = w - d
                if right_window_start_w - kernel_half < 0:
                    current_similarity = -1
                else:
                    right_window = right_img[h-kernel_half:h+kernel_half+1, right_window_start_w-kernel_half:right_window_start_w+kernel_half+1].flatten()
                    current_similarity = cosine_similarity(left_window, right_window)

                if current_similarity > max_similarity:
                    max_similarity = current_similarity
                    best_disparity = d
            depth[h, w] = int(best_disparity * scale_factor)
    if save_result:
        cv2.imwrite('depth_map_cosine_similarity.png', depth)
        color_map  =cv2.applyColorMap(depth, cv2.COLORMAP_JET)
        cv2.imwrite('depth_map_cosine_similarity_color.png', color_map)
    return depth

In [ ]:
left_img_path = "..\\data\\Aloe\\Aloe_left_1.png"
right_img_path = "..\\data\\Aloe\\Aloe_right_2.png"
disparity_range = 64
kernel_size = 3

window_based_result_cosine = window_based_matching_cosine_similarity(
    left_img_path,
    right_img_path,
    disparity_range=disparity_range,
    kernel_size=kernel_size,
    save_result=True
)